In [ ]:
# Setup: connect Google Drive and open the project folder (works in Google Colab)
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/NovelMate_ML_Project"
except ImportError:                      # running on your own computer instead of Colab
    PROJECT_DIR = "."
if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError("Upload the NovelMate_ML_Project folder to My Drive first (see README.md).")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.getcwd())

In [ ]:
!pip install -q fastapi "uvicorn[standard]"

In [ ]:
import importlib, threading, time
import requests, uvicorn

if "server" in globals():                       # stop an earlier run if this cell is executed again
    server.should_exit = True
    time.sleep(1)

import app as novel_app
importlib.reload(novel_app)
server = uvicorn.Server(uvicorn.Config(novel_app.app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

base = "http://127.0.0.1:8000"
info = requests.get(f"{base}/api/info").json()
print("Regression models    :", list(info["models"]["regression"].values()))
print("Classification models:", list(info["models"]["classification"].values()))

from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=1000)       # localhost links do not open from Colab

In [ ]:
sample = {"original_publication_year": 2000, "books_count": 50, "ratings_count": 50000,
          "work_ratings_count": 60000, "work_text_reviews_count": 3000, "language_code": "eng"}
for task in ("regression", "classification"):
    r = requests.post(f"{base}/compare/{task}", json=sample)
    print(task, r.status_code, [(x["name"], x.get("predicted_average_rating", x.get("category"))) for x in r.json().get("results", [])])
print("Invalid input ->", requests.post(f"{base}/compare/regression", json={**sample, "ratings_count": -5}).status_code, "(expected 422)")

In [ ]:
import joblib, shutil
import pandas as pd
from ml_utils import INFO_FILE, FEATURES, TARGET

info = joblib.load(INFO_FILE)
reg = pd.DataFrame(info["metrics"]["regression"])
clf = pd.DataFrame(info["metrics"]["classification"])
ranges = info["class_ranges"]

def md_table(df):
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

reg_text = clf_text = "No model trained yet."
if len(reg):
    best = reg.loc[reg["r2"].idxmax()]
    reg_text = (md_table(reg.drop(columns="algorithm").round(4).rename(columns={"name": "Model", "r2": "R2", "mae": "MAE", "rmse": "RMSE"}))
                + f"\n\nBest regression model: **{best['name']}** (R2 = {best['r2']:.4f}, MAE = {best['mae']:.4f}).")
    if best["r2"] < 0.3:
        reg_text += " The scores are modest, which is expected: the available fields say little about how much readers like a book."
if len(clf):
    best = clf.loc[clf["accuracy"].idxmax()]
    clf_text = (md_table(clf.drop(columns="algorithm").round(4).rename(columns={"name": "Model", "accuracy": "Accuracy", "f1": "F1 (weighted)"}))
                + f"\n\nBest classification model: **{best['name']}** (accuracy = {best['accuracy']:.4f}; random guessing is about 0.33).")

report = f"""# NovelMate: Brief Report

## 1. Objective
Predict the rating of a novel (regression) and its rating group (classification) with several machine learning models,
and make them usable through a FastAPI backend and an HTML/CSS/JavaScript frontend.

## 2. Dataset
Goodbooks-10k `books.csv`; {info['dataset_rows']} novels remained after cleaning.
Features: {', '.join(FEATURES)}.
Regression target: `{TARGET}`. Classification target: rating group (Low: {ranges['Low']}, Medium: {ranges['Medium']}, High: {ranges['High']}).
`ratings_1` to `ratings_5` were not used because they would reveal the rating (data leakage).

## 3. Preprocessing (`Data_Processing.ipynb`, `ml_utils.py`)
- Removed rows without a valid rating; negative counts treated as missing.
- Median imputation for numbers, "unknown" for a missing language.
- Log transform and scaling of the skewed counts, scaling of the year, one-hot encoding of the language.
- One fixed 80/20 train-test split (random_state = 42, stratified), shared by all models.

## 4. Models (one notebook each)
Regression: Linear Regression, Decision Tree, Random Forest, SVR.
Classification: Logistic Regression, Decision Tree (Gini), Decision Tree (Entropy), Random Forest, SVM, Naive Bayes.
Each model is a scikit-learn pipeline (preprocessing + model) saved with joblib in `saved_models/`.

## 5. Results on the test set
### Regression
{reg_text}

### Classification
{clf_text}

## 6. Application
- `app.py` (FastAPI): endpoints to predict with one model or compare all models, with input validation.
- `static/html/index.html` and `static/css/style.css`: form, model selector, comparison table and metrics table; it calls the backend with `fetch`.

## 7. Conclusion and limitations
The project covers preprocessing, training, evaluation, model saving, serving and comparison of multiple algorithms.
The models use only numeric metadata and language, so they cannot capture plot, writing quality or genre.
"""
Path("REPORT.md").write_text(report, encoding="utf-8")
print(report[:600], "...")

In [ ]:
# Zip the project (without cache folders) and download it
for cache in Path(".").rglob("__pycache__"):
    shutil.rmtree(cache, ignore_errors=True)
zip_path = shutil.make_archive("/content/NovelMate_ML_Project_final", "zip", root_dir=Path(".").resolve().parent, base_dir=Path(".").resolve().name)
print("Created", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
from google.colab import files
files.download(zip_path)